In [1]:
# CELL 1 — Install
!pip install -q unsloth
!pip install -q "trl>=0.15.0" "datasets" "openai" "httpx" "matplotlib" "peft" "accelerate"
!pip install -q mergekit
import torch, transformers, trl, unsloth, peft
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"trl: {trl.__version__}")
print(f"unsloth: {unsloth.__version__}")
print(f"peft: {peft.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — change runtime to T4'}")
print("Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

/tmp/ipykernel_1187/3208828317.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import torch, transformers, trl, unsloth, peft


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
torch: 2.10.0+cu128
transformers: 5.5.0
trl: 0.24.0
unsloth: 2026.4.6
peft: 0.18.1
GPU: Tesla T4
Done


In [2]:
# CELL 2 — Clone repo and setup
import os, sys

!git clone https://github.com/RAK2315/ml-debug-env.git
os.chdir('ml-debug-env')
sys.path.insert(0, 'server')
sys.path.insert(0, '.')

os.environ['GROQ_API_KEY'] = os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'
os.environ['PYTHON_EXEC'] = '/usr/bin/python3'

print('Working dir:', os.getcwd())
print('Done')

Cloning into 'ml-debug-env'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 173 (delta 87), reused 131 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 651.00 KiB | 23.25 MiB/s, done.
Resolving deltas: 100% (87/87), done.
Working dir: /content/ml-debug-env
Done


In [3]:
# CELL 3 — Test environment imports
from bug_generator import get_scenario, execute_tool, ALL_TASKS, AVAILABLE_TOOLS
from grader import grade

s = get_scenario('shape_mismatch', seed=42)
print('Alert:', s.alert)
print('Tasks:', ALL_TASKS)
print('Tools:', AVAILABLE_TOOLS)
print('Done')

Alert: Training job crashed immediately. No epochs completed. Exit code 1.
Tasks: ['shape_mismatch', 'training_collapse', 'data_leakage', 'wrong_device', 'gradient_not_zeroed', 'missing_eval_mode', 'compound_shape_device', 'compound_leakage_eval']
Tools: ['run_code', 'get_traceback', 'inspect_gradients', 'print_shapes', 'view_source']
Done


In [4]:
# CELL 4 — Load model with Unsloth
import unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()
print('Model loaded OK')

==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.6 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
Model loaded OK


In [5]:
# CELL 5 — System prompt, reward function, parse_action
import json

TRAIN_SYSTEM = """You are an expert ML engineer. Debug the broken PyTorch script.
Respond with JSON only:
{"bug_type": "<shape_mismatch|training_collapse|data_leakage|wrong_device|gradient_not_zeroed|missing_eval_mode|other>", "diagnosis": "<explanation>", "fixed_code": "<complete fixed script>"}
No markdown. No text outside JSON."""

def make_train_prompt(task_id, seed=42):
    scenario = get_scenario(task_id, seed=seed)
    return [
        {"role": "system", "content": TRAIN_SYSTEM},
        {"role": "user", "content": (
            f"Broken script:\n```python\n{scenario.buggy_code}\n```\n\n"
            f"Error/failure:\n{scenario.error_output}\n\n"
            f"Return JSON fix only."
        )}
    ], scenario

def parse_action(text):
    text = text.strip()
    # try direct parse first
    try:
        return json.loads(text)
    except:
        pass
    # strip markdown fences
    if '```' in text:
        parts = text.split('```')
        for part in parts:
            part = part.strip()
            if part.startswith('json'):
                part = part[4:].strip()
            if part.startswith('{'):
                try:
                    return json.loads(part)
                except:
                    continue
    # find first { ... } block
    try:
        start = text.index('{')
        end = text.rindex('}') + 1
        return json.loads(text[start:end])
    except:
        pass
    return {}

def compute_reward(completion, task_id, seed=42):
    try:
        parsed = parse_action(completion)
        action_type = parsed.get('action_type', 'fix')
        if action_type == 'inspect':
            return 0.0
        bug_type = parsed.get('bug_type', 'other')
        diagnosis = parsed.get('diagnosis', '')
        fixed_code = parsed.get('fixed_code', '')
        if not fixed_code or len(fixed_code) < 50:
            return 0.0
        scenario = get_scenario(task_id, seed=seed)
        result = grade(
            action_bug_type=bug_type,
            action_diagnosis=diagnosis,
            fixed_code=fixed_code,
            scenario=scenario,
        )
        return float(result.score)
    except Exception as e:
        print(f"  [reward error] {e}")
        return 0.0

# Quick test
messages, scenario = make_train_prompt('shape_mismatch', seed=42)
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print('Prompt length:', len(prompt))
print('Done')

Prompt length: 1598
Done


In [6]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CELL 6 — GRPO training loop (fixed)
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('transformers').setLevel(logging.ERROR)
import torch
from torch.optim import AdamW
import random

optimizer = AdamW(model.parameters(), lr=5e-6)
reward_log = []
MAX_STEPS = 200
NUM_GENERATIONS = 4
MIN_COMPLETION_CHARS = 50   # was 100
MIN_REWARD = 0.015          # was 0.05

TRAIN_TASKS = ['shape_mismatch', 'training_collapse', 'wrong_device', 'gradient_not_zeroed']

def generate_completion(messages):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to('cuda')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=True,
            temperature=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

dataset = []
for i in range(50):
    for task_id in TRAIN_TASKS:
        dataset.append({'task_id': task_id, 'seed': i})
random.shuffle(dataset)

prompt_text_cache = {}

step = 0
skipped = 0
for episode in dataset:
    if step >= MAX_STEPS:
        break

    task_id = episode['task_id']
    seed = episode['seed']
    messages, scenario = make_train_prompt(task_id, seed=seed)

    completions, rewards = [], []
    for _ in range(NUM_GENERATIONS):
        completion = generate_completion(messages)
        # Skip short/garbage outputs — do not train on them
        if len(completion.strip()) < MIN_COMPLETION_CHARS:
            skipped += 1
            continue
        r = compute_reward(completion, task_id, seed=seed)
        # Skip wrong-bug-type noise
        if r <= MIN_REWARD:
            skipped += 1
            continue
        completions.append(completion)
        rewards.append(r)

    if len(rewards) < 2:
        # Not enough valid completions to compute meaningful advantage
        avg_reward = float(sum(rewards) / len(rewards)) if rewards else 0.0
        reward_log.append({'step': step, 'reward': avg_reward})
        print(f'Step {step:3d} | {task_id:25s} | skipped (valid={len(rewards)}) | avg={avg_reward:.3f}')
        step += 1
        continue

    rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
    avg_reward = float(rewards_tensor.mean())
    reward_log.append({'step': step, 'reward': avg_reward})

    if rewards_tensor.std() < 1e-6:
        print(f'Step {step:3d} | {task_id:25s} | {[f"{r:.2f}" for r in rewards]} | avg={avg_reward:.3f} [no variance]')
        step += 1
        continue

    mean_r = rewards_tensor.mean()
    std_r = rewards_tensor.std() + 1e-8
    advantages = (rewards_tensor - mean_r) / std_r

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Train on ALL valid completions weighted by their advantage (proper GRPO)
    model.train()
    total_loss = torch.tensor(0.0, device='cuda')
    trained_on = 0
    for i, (completion, adv) in enumerate(zip(completions, advantages.tolist())):
        full_text = prompt_text + completion
        inputs = tokenizer(
            full_text, return_tensors='pt', truncation=True, max_length=1500
        ).to('cuda')
        outputs = model(**inputs, labels=inputs['input_ids'])
        total_loss = total_loss + (-adv * outputs.loss)
        trained_on += 1

    if trained_on > 0:
        loss = total_loss / trained_on
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    print(f'Step {step:3d} | {task_id:25s} | {["{:.2f}".format(r) for r in rewards]} | avg={avg_reward:.3f}')
    step += 1

print(f'Done. Steps: {step}, total skipped generations: {skipped}')


Step   0 | wrong_device              | ['0.23', '0.26'] | avg=0.245
Step   1 | gradient_not_zeroed       | skipped (valid=0) | avg=0.000
Step   2 | wrong_device              | ['0.28', '0.30'] | avg=0.290
Step   3 | gradient_not_zeroed       | skipped (valid=0) | avg=0.000
Step   4 | gradient_not_zeroed       | ['0.32', '0.20'] | avg=0.260
Step   5 | training_collapse         | ['0.23', '0.40'] | avg=0.315
Step   6 | gradient_not_zeroed       | skipped (valid=0) | avg=0.000
Step   7 | training_collapse         | ['0.32', '0.32', '0.48'] | avg=0.372
Step   8 | gradient_not_zeroed       | skipped (valid=0) | avg=0.000
Step   9 | gradient_not_zeroed       | skipped (valid=1) | avg=0.200
Step  10 | wrong_device              | ['0.35', '0.26'] | avg=0.305
Step  11 | shape_mismatch            | ['0.46', '0.46'] | avg=0.460 [no variance]
Step  12 | wrong_device              | ['0.26', '0.20', '0.26', '0.30'] | avg=0.256
Step  13 | gradient_not_zeroed       | skipped (valid=0) | avg=0.000
Step

In [1]:
# CELL 7 — Plot reward curve (dual: raw + percentage)
import matplotlib.pyplot as plt
import numpy as np

steps = [r['step'] for r in reward_log]
rewards = [r['reward'] for r in reward_log]

window = 8
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

initial_avg = np.mean(rewards[:8])
final_avg = np.mean(rewards[-8:])
improvement_pct = ((final_avg - initial_avg) / initial_avg * 100) if initial_avg > 0 else 0

# Trend line
z = np.polyfit(steps, rewards, 1)
p = np.poly1d(z)
trend = p(steps)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

# --- Plot 1: Raw reward (0-0.99) ---
ax1.plot(steps, rewards, alpha=0.3, color='steelblue', label='Raw reward')
ax1.plot(steps[window-1:], smoothed, color='steelblue', linewidth=2.5, label=f'Smoothed (window={window})')
ax1.plot(steps, trend, color='red', linestyle='--', alpha=0.8, label=f'Trend ({"+" if z[0]>=0 else ""}{z[0]*100:.3f}/step)')
ax1.axhline(y=initial_avg, color='orange', linestyle=':', alpha=0.7, label=f'Initial avg: {initial_avg:.3f}')
ax1.axhline(y=final_avg, color='green', linestyle=':', alpha=0.7, label=f'Final avg: {final_avg:.3f}')
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Reward (0 - 0.99)')
ax1.set_title('ML Debug Env — GRPO Training\nQwen2.5-1.5B-Instruct + LoRA (4bit)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1.0)

# --- Plot 2: Percentage scale ---
rewards_pct = [r * 100 for r in rewards]
smoothed_pct = np.convolve(rewards_pct, np.ones(window)/window, mode='valid')
trend_pct = trend * 100

ax2.plot(steps, rewards_pct, alpha=0.3, color='darkorange', label='Raw reward %')
ax2.plot(steps[window-1:], smoothed_pct, color='darkorange', linewidth=2.5, label=f'Smoothed (window={window})')
ax2.plot(steps, trend_pct, color='red', linestyle='--', alpha=0.8, label=f'Trend ({"+" if z[0]>=0 else ""}{z[0]*100:.2f}%/step)')
ax2.axhline(y=initial_avg*100, color='orange', linestyle=':', alpha=0.7, label=f'Initial: {initial_avg*100:.1f}%')
ax2.axhline(y=final_avg*100, color='green', linestyle=':', alpha=0.7, label=f'Final: {final_avg*100:.1f}%')
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Reward %')
ax2.set_title(f'ML Debug Env — GRPO Training\n+{improvement_pct:.0f}% improvement ({initial_avg*100:.1f}% → {final_avg*100:.1f}%)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('reward_curve_dual.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Initial reward (first 8 steps): {initial_avg:.3f} ({initial_avg*100:.1f}%)')
print(f'Final reward (last 8 steps):    {final_avg:.3f} ({final_avg*100:.1f}%)')
print(f'Improvement: {final_avg - initial_avg:+.3f} ({improvement_pct:+.1f}%)')

NameError: name 'reward_log' is not defined

In [ ]:
# CELL 8 — Before/after test
import torch
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def make_prompt(task_id, seed=42):
    scenario = get_scenario(task_id, seed=seed)
    return [
        {"role": "system", "content": TRAIN_SYSTEM},
        {"role": "user", "content": (
            f"Broken script:\n```python\n{scenario.buggy_code}\n```\n\n"
            f"Error/failure:\n{scenario.error_output}\n\n"
            f"Return JSON fix only."
        )}
    ], scenario

TEST_TASK = 'compound_leakage_eval'
messages, scenario = make_prompt(TEST_TASK, seed=99)
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
reward = compute_reward(response, TEST_TASK, seed=99)
print(f'Task: {TEST_TASK}')
print(f'Score: {reward:.3f}')
print(f'Response: {response[:400]}')

In [ ]:
# CELL 9 — Save everything
import json

# Save reward log
with open('reward_log.json', 'w') as f:
    json.dump(reward_log, f)

# Save baseline score
baseline = {'task': 'compound_leakage_eval', 'seed': 99, 'score': 0.0, 'response': 'inspect:print_shapes'}
with open('baseline_result.json', 'w') as f:
    json.dump(baseline, f)

# Download reward curve
from google.colab import files
files.download('reward_curve.png')

print('Saved. Download reward_curve.png from the files panel.')

In [ ]:
import importlib
import grader
importlib.reload(grader)
from grader import grade

# Now rerun test
from bug_generator import get_scenario

test_cases = [
    ('shape_mismatch', 42),
    ('training_collapse', 42),
    ('wrong_device', 42),
    ('gradient_not_zeroed', 42),
]

for task_id, seed in test_cases:
    scenario = get_scenario(task_id, seed=seed)
    bad_result = grade('other', 'bad fix', scenario.buggy_code, scenario)
    good_result = grade(task_id, 'correct diagnosis', scenario.buggy_code, scenario)
    print(f"{task_id:25s} | bad_fix={bad_result.score:.2f} | correct_type={good_result.score:.2f}")

shape_mismatch            | bad_fix=0.01 | correct_type=0.26
training_collapse         | bad_fix=0.01 | correct_type=0.40
wrong_device              | bad_fix=0.01 | correct_type=0.26
gradient_not_zeroed       | bad_fix=0.01 | correct_type=0.66


In [ ]:
import importlib
import bug_generator
importlib.reload(bug_generator)
import grader
importlib.reload(grader)
from grader import grade
from bug_generator import get_scenario

scenario = get_scenario('gradient_not_zeroed', seed=42)
result = grade('gradient_not_zeroed', 'test', scenario.buggy_code, scenario)
print(f"Score: {result.score}")
print(f"Feedback: {result.feedback[:300]}")